# OS: Optimal Site

**Atlas source reading copy · browser-local**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Spatial-Analysis/blob/88007a32ab0ac970469af4b0c118209325a1989f/notebooks/26_OS_optimal-site.ipynb)



Local calculations and bundled inputs; first browser runtime/package download still needs connectivity. Static inspection is not execution certification.

**Provenance:** `notebooks/26_OS_optimal-site.ipynb` at `88007a32ab0ac970469af4b0c118209325a1989f`; original SHA-256 `9605e03fb1a7477c860c601053cc3e52291a1803b1ca0e614348f52a54e9f300`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# OS: Optimal Site

**Category:** Network Analysis

**Concept:** Chooses facility sites that best serve demand.

This notebook is part of a JupyterLite learning lab inspired by the GISGeography *Periodic Table for Spatial Analysis*. It uses synthetic data so it can run entirely in the browser.

## Learning goals

- Recognize what **Optimal Site** tries to answer in a geospatial workflow.
- Run a minimal Python demonstration using points, networks, or rasters.
- Modify one parameter and explain how the result changes.


In [ ]:
# Shared setup for browser-friendly geospatial examples.
# These notebooks avoid heavy compiled GIS dependencies so they run well in JupyterLite/Pyodide.
import math
import random
from collections import defaultdict, deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(7)
np.random.seed(7)

def make_points(n=60, centers=((2, 2), (7, 5)), spread=0.85):
    """Create clustered x/y observations with simple attributes."""
    pts = []
    for i in range(n):
        cx, cy = centers[i % len(centers)]
        x = np.random.normal(cx, spread)
        y = np.random.normal(cy, spread)
        pts.append({'id': i, 'x': x, 'y': y, 'group': chr(65 + (i % len(centers))), 'value': 10 + 2*x + y + np.random.normal(0, 1)})
    return pd.DataFrame(pts)

def grid_surface(nx=80, ny=60):
    """Synthetic elevation-like raster with hills and noise."""
    x = np.linspace(0, 10, nx)
    y = np.linspace(0, 8, ny)
    X, Y = np.meshgrid(x, y)
    Z = 70*np.exp(-((X-3.0)**2 + (Y-5.5)**2)/5.0) + 45*np.exp(-((X-7.5)**2 + (Y-2.5)**2)/3.0)
    Z += 4*np.sin(X*1.7) + 3*np.cos(Y*2.1)
    return X, Y, Z

def plot_points(df, title='Point layer', value_col='value'):
    fig, ax = plt.subplots(figsize=(6, 5))
    s = ax.scatter(df['x'], df['y'], c=df[value_col], s=55)
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal', adjustable='box')
    fig.colorbar(s, ax=ax, label=value_col)
    plt.show()

def plot_raster(Z, title='Raster surface'):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(Z, origin='lower', extent=[0, 10, 0, 8])
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    fig.colorbar(im, ax=ax)
    plt.show()

def distance(a, b):
    return math.hypot(a[0]-b[0], a[1]-b[1])

def nearest_neighbor(df, facilities):
    rows = []
    for _, r in df.iterrows():
        dists = [(name, distance((r.x, r.y), xy)) for name, xy in facilities.items()]
        name, d = min(dists, key=lambda t: t[1])
        rows.append((name, d))
    out = df.copy()
    out['nearest'] = [r[0] for r in rows]
    out['distance'] = [r[1] for r in rows]
    return out

def moving_average_2d(A, radius=1):
    B = np.zeros_like(A, dtype=float)
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            r0, r1 = max(0, i-radius), min(A.shape[0], i+radius+1)
            c0, c1 = max(0, j-radius), min(A.shape[1], j+radius+1)
            B[i, j] = A[r0:r1, c0:c1].mean()
    return B


## Run the mini-lab

The next cell creates a small synthetic geospatial problem. The example is deliberately compact: focus on the concept, then adapt it to your own data or a richer GIS stack later.


In [ ]:
# OS: Optimal Site mini-lab
nodes = {'A': (1, 1), 'B': (3, 2), 'C': (5, 1.5), 'D': (7, 3), 'E': (8, 6), 'F': (4, 6), 'G': (2, 5)}
edges = [('A','B',2.2), ('B','C',2.1), ('C','D',2.5), ('D','E',3.0), ('B','G',3.2), ('G','F',2.2), ('F','E',4.2), ('C','F',4.5)]

def shortest_path(start, goal):
    graph = defaultdict(list)
    for a, b, w in edges:
        graph[a].append((b, w)); graph[b].append((a, w))
    dist = {start: 0}
    prev = {}
    q = set(nodes)
    while q:
        u = min(q, key=lambda n: dist.get(n, float('inf')))
        q.remove(u)
        if u == goal: break
        for v, w in graph[u]:
            alt = dist[u] + w
            if alt < dist.get(v, float('inf')):
                dist[v] = alt; prev[v] = u
    path = [goal]
    while path[-1] != start:
        path.append(prev[path[-1]])
    return list(reversed(path)), dist[goal]

path, cost = shortest_path('A', 'E')
fig, ax = plt.subplots(figsize=(6, 5))
for a, b, w in edges:
    ax.plot([nodes[a][0], nodes[b][0]], [nodes[a][1], nodes[b][1]], linewidth=1)
for n, (x, y) in nodes.items():
    ax.scatter(x, y, s=80); ax.text(x+0.08, y+0.08, n)
for a, b in zip(path[:-1], path[1:]):
    ax.plot([nodes[a][0], nodes[b][0]], [nodes[a][1], nodes[b][1]], linewidth=4)
ax.set_title(f'OS: Optimal Site - least-cost path A to E = {cost:.1f}')
ax.set_aspect('equal'); plt.show()

# Origin-destination style matrix for all nodes.
rows = []
for o in nodes:
    for d in nodes:
        if o != d:
            _, c = shortest_path(o, d)
            rows.append({'origin': o, 'destination': d, 'cost': round(c, 2)})
pd.DataFrame(rows).head(12)


## Reflection prompts

1. What real dataset could you use to practice **Optimal Site**?
2. Which parameter in the code has the biggest effect on the result?
3. What would make this analysis misleading if the input data were noisy, biased, or in the wrong coordinate reference system?

## Next steps

- Compare this notebook with related elements in **Network Analysis**.
- Replace the synthetic coordinates with a CSV of real locations.
- Add a short markdown note explaining when you would use this method in public health, environment, emergency response, planning, or transportation.
